In [2]:
import pandas as pd
import json
import re
import anthropic
import os
from dotenv import load_dotenv

load_dotenv()
client = anthropic.Anthropic(api_key=os.environ.get("ANTHROPIC_API_KEY"))

print("all good")

all good


In [3]:

def generate_tweets(n):
    prompt = f"""Generate {n} synthetic tweets as a JSON array. Each tweet must conform exactly to this schema:

{{
  "id": "<unique numeric string>",
  "text": "<tweet text, max 280 chars>",
  "created_at": "<ISO 8601 timestamp>",
  "entities": {{
    "urls": [{{"expanded_url": "<full url or empty list>"}}],
    "mentions": [{{"username": "<handle without @>"}}],
    "hashtags": [{{"tag": "<hashtag without #>"}}]
  }},
  "author": {{
    "id": "<unique numeric string>",
    "username": "<twitter handle>",
    "name": "<display name>",
    "verified": <true|false>,
    "created_at": "<ISO 8601 timestamp>",
    "public_metrics": {{
      "followers_count": <int>,
      "following_count": <int>,
      "tweet_count": <int>
    }}
  }},
  "public_metrics": {{
    "retweet_count": <int>,
    "reply_count": <int>,
    "like_count": <int>,
    "quote_count": <int>
  }},
  "label": <1 for scam, 0 for legitimate, -1 for ambiguous>
}}

Generate in this distribution:
- 40% scam (label: 1): crypto giveaway scams, impersonation of Elon Musk/Vitalik/CZ,
  "send ETH get 2x back" schemes, phishing links, urgent limited-time offers.
  Fresh accounts: low followers, high following, young account age, suspicious usernames.
- 40% legitimate (label: 0): real crypto discussion, news sharing, price commentary,
  developer posts. Aged profiles, balanced follower ratios, occasionsally verified.
- 20% ambiguous (label: -1): aggressive but legitimate marketing, new accounts posting
  real content, promotional language that isn't technically a scam.

Make metadata internally consistent — a 3-day-old account should have low tweet_count,
scam tweets should have inflated likes but near-zero replies.
Vary writing style, don't repeat phrases across tweets.
Return ONLY the raw JSON array, no explanation or markdown."""

    return prompt


In [5]:
try: 
    with open("data/tweets.json", "r") as f:
        tweets = json.load(f)
    print(f"loaded {len(tweets)} existing tweets")
except FileNotFoundError:
    tweets = []

for _ in range(10):
    message = client.messages.create(
        model="claude-opus-4-5",
        max_tokens=8096,
        messages=[{"role": "user", "content": generate_tweets(10)}]
    )
    response_text = message.content[0].text
    response_text = response_text.strip().removeprefix("```json").removesuffix("```").strip()
    tweets += json.loads(response_text)
    print(f"Total so far: {len(tweets)}")

with open("data/tweets.json", "w") as f:
    json.dump(tweets, f, indent=2)
    

print(f"Final Total: {len(tweets)}")

loaded 200 existing tweets
Total so far: 210
Total so far: 220
Total so far: 230
Total so far: 240
Total so far: 250
Total so far: 260
Total so far: 270
Total so far: 280
Total so far: 290
Total so far: 300
Final Total: 300


In [14]:
print(response_text[21080:21150])

": {
      "retweet_count": 1


In [ ]:
with open("data/tweets.json", "w") as f:
    tweets = json.load(f)